In [1]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Standardisert treningsoppsett for endometriekreft-tumorsegmentering
– støtter 1-3 modaliteter (VIBE, T2, ADC)
"""

# ==========================================================
# Standard library
# ==========================================================
import os
import datetime
import warnings
import shutil

# ==========================================================
# Numerical / data
# ==========================================================
import numpy as np
import pandas as pd

# ==========================================================
# PyTorch / TorchIO
# ==========================================================
import torch
from torch.utils.data import DataLoader
import torchio as tio

# ==========================================================
# MONAI
# ==========================================================
from monai.networks.nets import UNet
from monai.losses import DiceCELoss, DiceLoss
from monai.networks.nets import SwinUNETR


# ==========================================================
# fastai
# ==========================================================
from fastai.data.core import DataLoaders
from fastai.learner import Learner, Metric
from fastai.callback.tracker import EarlyStoppingCallback, SaveModelCallback
from fastai.callback.fp16 import MixedPrecision

# ==========================================================
# Project-specific
# ==========================================================
import params
from datasetgenerator import datasetgenerator_bergen
from utils import *


2026-01-02 09:05:13.345468: I tensorflow/tsl/cuda/cudart_stub.cc:28] Could not find cuda drivers on your machine, GPU will not be used.
2026-01-02 09:05:13.406805: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2026-01-02 09:05:14.872194: W tensorflow/compiler/tf2tensorrt/utils/py_utils.cc:38] TF-TRT Warning: Could not find TensorRT


In [2]:
# Hvilke modaliteter som skal brukes i dette eksperimentet
#selected_modalities = ["T2", "ADC"]
#selected_modalities = ["T2"]
#selected_modalities = ["vibe2min", "T2", "ADC"]
selected_modalities = ["vibe2min"]

# Hvilket bilde som brukes som referanse
#reference = "T2"
reference = "vibe2min"


# ==============================================================
# 1. Les inn og filtrer datasett
# ==============================================================
df = pd.read_csv(params.pathlistvalid, sep=';').groupby("subj", as_index=False).first()

# Which columsn to require
require = [params.modalities[m]["col"] for m in selected_modalities]
df = datasetgenerator_bergen(df, require)

# Only data sets with manual masks
df = df.loc[df.dataset == 'man'].reset_index(drop=True)
print(f"Antall datasett med manuelle masker: {len(df)}")

# ==============================================================
# 2. Bygg dynamiske paths for bilder og masker
# ==============================================================

df["imgpath"] = [
    build_image_path(
        s,
        selected_modalities,
        params.prepathnifti,
        reference,
        params.modalities
    )
    for s in df.subj
]

df["pathmask"] = [
    build_mask_path(s, m, params.prepathnifti, reference)
    for s, m in zip(df.subj, df.pathmask)
]
df.head(3)

587 datasett tilfredsstiller betingelsene
Antall datasett med manuelle masker: 273


,subj,pathvibe2minDicom,pathT2Dicom,pathADCDicom,pathJADNifti,pathKWLNifti,pathVerifiedMLNifti,pathJADMLNifti,pathmask,dataset,imgpath
0,11,fl3d_vibe_tra_2mm_2min,t2_tse_skra_tra_p2,ep2d_diff_adc,011segmentedJulie.nii.gz,None,None,None,/raid/erlend/GynKreft/Data-EC/Nifti/EC011/registered/011segmentedJulie-2-vibe2min-header.nii.gz,man,/raid/erlend/GynKreft/Data-EC/Nifti/EC011/unregistered/vibe2min.nii.gz
1,17,fl3d_vibe_tra_2mm_2min,t2_tse_skra_tra_p2,ep2d_diff_adc,017segmentedJulie.nii.gz,None,None,None,/raid/erlend/GynKreft/Data-EC/Nifti/EC017/registered/017segmentedJulie-2-vibe2min-header.nii.gz,man,/raid/erlend/GynKreft/Data-EC/Nifti/EC017/unregistered/vibe2min.nii.gz
2,20,fl3d_vibe_tra_2mm_2min,t2_tse_skra_tra_p2,ep2d_diff_adc,020segmentedJulie.nii.gz,None,None,None,/raid/erlend/GynKreft/Data-EC/Nifti/EC020/registered/020segmentedJulie-2-vibe2min-header.nii.gz,man,/raid/erlend/GynKreft/Data-EC/Nifti/EC020/unregistered/vibe2min.nii.gz


In [3]:
from tqdm import tqdm
print("🔍 Beregner tumorvolum for alle masker ...")
df["tumorsize"] = [compute_tumor_volume(p) for p in tqdm(df["pathmask"].values)]
print(f"✅ Volum beregnet for {df['tumorsize'].notna().sum()} pasienter.")


🔍 Beregner tumorvolum for alle masker ...


100%|████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 273/273 [00:20<00:00, 13.59it/s]

✅ Volum beregnet for 273 pasienter.


In [4]:
# ==============================================================
# 3. Split datasett i train / val / test
# ==============================================================
# Del datasettet i trenings-, validerings- og test-sett.
# - test_fraction definerer hvor stor andel som holdes av til test.
# - validation_fraction brukes til å trekke ut et subset av trening til validering.

# Lag kategorier basert på tumorstørrelse (kvartiler)
df["tumorsize_cat"] = pd.qcut(
    df["tumorsize"],
    q=4,  # antall grupper (4=kvartiler)
    labels=["Q1_Smallest", "Q2", "Q3", "Q4_Largest"]
)

# Fjern NaN (masker som ikke kunne leses)
df = df.dropna(subset=["tumorsize_cat"]).reset_index(drop=True)
from sklearn.model_selection import train_test_split

test_fraction = 0.2
random_state = 42
validation_fraction = 0.1

dftrain, dftest = train_test_split(
    df,
    test_size=test_fraction,
    stratify=df["tumorsize_cat"],
    random_state=random_state
)

# Lag valideringssplit som før
val_idx = dftrain.sample(
    frac=validation_fraction,
    random_state=random_state
).index
dftrain["isval"] = False
dftrain.loc[val_idx, "isval"] = True

# Skriv ut fordeling
print(f"\n🧠 Modaliteter: {params.modalities}")
print(f"Train: {len(dftrain)},  Val: {dftrain['isval'].sum()},  Test: {len(dftest)}")

# Fordeling av tumorvolum per gruppe
print("\n📊 Tumorvolum-fordeling (ml):")
print(df.groupby("tumorsize_cat")["tumorsize"].describe()[["min", "max", "mean"]])



🧠 Modaliteter: {'vibe2min': {'col': 'pathvibe2minDicom', 'file': 'vibe2min.nii.gz'}, 'T2': {'col': 'pathT2Dicom', 'file': 'T2.nii.gz'}, 'ADC': {'col': 'pathADCDicom', 'file': 'ADC.nii.gz'}}
Train: 218,  Val: 22,  Test: 55

📊 Tumorvolum-fordeling (ml):
                     min         max       mean
tumorsize_cat                                  
Q1_Smallest     0.078970    3.340787   1.577923
Q2              3.362532    7.872995   5.636550
Q3              7.949676   19.056533  12.337547
Q4_Largest     19.280329  679.400909  62.928205


In [5]:
warnings.filterwarnings("ignore", message="Using TorchIO images without a torchio.SubjectsLoader")

# ==========================================================
# 4. PARAMETERE
# ==========================================================
GPU_ID         = 2
batch_size     = 6
img_size       = 192
n_epochs       = 200
learning_rate  = 1e-3
monitor_metric = "dice"
patience       = 40
weight_decay   = 1e-5
dropout        = 0.1
modelname      = "UNet"

assert torch.cuda.is_available()
torch.cuda.set_device(GPU_ID)
device = torch.device(f"cuda:{GPU_ID}")
print(f"Bruker GPU {device}")


# ==========================================================
# 5. Dataloaders
# ==========================================================
train_dl = get_dataloader(dftrain[dftrain.isval == False], batch_size, img_size, selected_modalities, augment=True)
val_dl   = get_dataloader(dftrain[dftrain.isval == True ], batch_size, img_size, selected_modalities, augment=False)

dls = DataLoaders(train_dl, val_dl, device=device)
dls.n_inp = 1

# ==========================================================
# 6. Modell
# ==========================================================
model = monai_unet_model(in_channels=len(selected_modalities)).to(device)

# ==========================================================
# 7. Metric name fix
# ==========================================================
metric = ThresholdedDice()
if not hasattr(metric, "name"):
    metric.__dict__["name"] = "dice"

# ==========================================================
# 8. Learner
# ==========================================================
learn = Learner(
    dls,
    model,
    loss_func=DiceLoss(sigmoid=True),
    metrics=[metric],
    cbs=[
        MixedPrecision(),
        SaveModelCallback(
            monitor=monitor_metric,     # "dice"
            comp=np.greater,
            fname="best_model"
        ),
        EarlyStoppingCallback(
            monitor=monitor_metric,
            comp=np.greater,
            patience=patience
        )
    ]
)

# ==========================================================
# 9. Trening
# ==========================================================
learn.fit_one_cycle(n_epochs, lr_max=learning_rate, wd=weight_decay)
learn.load("best_model")

# ==========================================================
# 10. Lagring av modell + settings
# ==========================================================
timestamp = datetime.datetime.now().strftime("%Y%m%d")
modalities_str = "_".join(selected_modalities)
basename = f"{modelname}_{modalities_str}_{params.version}_{timestamp}"

save_dir = os.path.join(params.prepathmodels, basename)
os.makedirs(save_dir, exist_ok=True)

model_path = os.path.join(save_dir, "model_best.pth")
torch.save(model.state_dict(), model_path)

save_training_artifacts(
    save_dir=save_dir,
    basename=basename,
    timestamp=timestamp,
    batch_size=batch_size,
    img_size=img_size,
    modalities=selected_modalities,
    train_df=dftrain[dftrain.isval == False],
    val_df=dftrain[dftrain.isval == True],
    test_df=dftest,
    model_path=model_path,
    modelname=modelname,
    dropout=dropout,
    lr=learning_rate,
    epochs=n_epochs,
    patience=patience,
)

# lagre notebook eller script
shutil.copy("train_UNet_Monai.ipynb", save_dir)
print(f"📑 Notebook lagret i {save_dir}")



Bruker GPU cuda:2


epoch,train_loss,valid_loss,dice,time
0,0.994640,0.994364,0.005466,02:47
1,0.994317,0.993935,0.006105,01:20
2,0.994033,0.993410,0.006734,01:18
3,0.993536,0.992888,0.007031,01:29
4,0.993125,0.992408,0.007289,01:19
5,0.992755,0.991842,0.007829,01:18
6,0.992298,0.991323,0.007738,01:18
7,0.991806,0.991119,0.007594,01:21
8,0.991546,0.991016,0.007519,01:22
9,0.991390,0.990920,0.007347,01:24


Better model found at epoch 0 with dice value: 0.005465628899401054.
Better model found at epoch 1 with dice value: 0.006105372274760157.
Better model found at epoch 2 with dice value: 0.006734113325364888.
Better model found at epoch 3 with dice value: 0.007030554523225874.
Better model found at epoch 4 with dice value: 0.007289174769539386.
Better model found at epoch 5 with dice value: 0.00782949716085568.
Better model found at epoch 11 with dice value: 0.008226176025345922.
Better model found at epoch 13 with dice value: 0.010152678412850946.
Better model found at epoch 15 with dice value: 0.010407784895505756.
Better model found at epoch 16 with dice value: 0.01415795087814331.
Better model found at epoch 17 with dice value: 0.034082457423210144.
Better model found at epoch 18 with dice value: 0.052210213616490364.
Better model found at epoch 20 with dice value: 0.05308400094509125.
Better model found at epoch 21 with dice value: 0.07400002842769027.
Better model found at epoch 22

KeyboardInterrupt: 

In [ ]:

# ==========================================================
# 11. Plotting
# ==========================================================
ax = learn.recorder.plot_loss()
fig = ax.figure
plt.show()
fig.savefig(os.path.join(save_dir, "loss_curve.png"), dpi=150, bbox_inches="tight")
plt.close(fig)


In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt


# ----------------------------------------------------------
# Plot én batch (image, GT, prediction)
# ----------------------------------------------------------
def show_max_mask_predictions(batch, predictions, dice_score, threshold=0.5):
    images, gts = batch
    images, gts, predictions = [x.cpu().numpy() for x in (images, gts, predictions)]

    num_images = images.shape[0]
    fig, axes = plt.subplots(num_images, 3, figsize=(20, 5*num_images))
    axes = np.array(axes).reshape(num_images, 3)

    for i in range(num_images):
        img = images[i, 0]
        gt  = gts[i, 0]
        pred_prob = predictions[i, 0]
        pred_mask = (pred_prob > threshold).astype(float)

        # Finn snittet med mest maske
        slice_sums = gt.sum(axis=(0, 1))
        z = slice_sums.argmax()

        # Original
        axes[i, 0].imshow(img[:, :, z], cmap='gray')
        axes[i, 0].set_title(f"Image {i}")
        axes[i, 0].axis('off')

        # Ground truth
        axes[i, 1].imshow(gt[:, :, z], cmap='gray')
        axes[i, 1].set_title("Ground Truth")
        axes[i, 1].axis('off')

        # Predicted mask med Dice
        axes[i, 2].imshow(pred_mask[:, :, z], cmap='gray', vmin=0, vmax=1)
        axes[i, 2].set_title(f"Pred Mask\nDice={dice_score[i]:.3f}")
        axes[i, 2].axis('off')

    plt.tight_layout()
    plt.show()


# ----------------------------------------------------------
# 🔥 **Samlet test-evalueringsfunksjon**
# ----------------------------------------------------------
def evaluate_test_set(model, dftest, batch_size, img_size, device, threshold=0.5):
    """
    Kjører hele evalueringspipen:
    - looper gjennom test-sett
    - predikerer masker
    - viser plot
    - beregner og printer Dice
    """

    test_dl = get_dataloader(dftest, batch_size, img_size, augment=False)
    model.eval()
    model.to(device)

    all_dice = []

    with torch.no_grad():
        for batch in test_dl:
            # batch = (images, masks, subj_ids) hvis du bruker subj; her ignorerer vi subj
            images, targets, *rest = batch
            images = images.to(device)
            targets = targets.to(device)

            # Sigmoid output
            preds = torch.sigmoid(model(images))

            # Dice
            batch_dice = compute_dice_score(preds, targets, apply_sigmoid=False, threshold=threshold)
            all_dice.extend(batch_dice)

            # Plot volumene
            show_max_mask_predictions((images, targets), preds, batch_dice, threshold=threshold)

    print("\n============================")
    print(f"🔥 Mean Dice: {np.mean(all_dice):.4f}")
    print("============================\n")

    return all_dice

device = torch.device("cuda:0")
dice_scores = evaluate_test_set(
    model=learn.model,
    dftest=dftest,
    batch_size=batch_size,
    img_size=img_size,
    device=device,
    threshold=0.5,
)
